# AutoML Clasificacion

Requirement Databricks Runtime 8.3 ML or above.

## 1. Configuración

In [0]:
%pip install --quiet databricks-automl-runtime
%pip install --quiet mlflow databricks-sdk
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "AutoML_customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("project_path", "/Workspace/Proyectos_Dev/databricks-medallion", "Ruta del proyecto")

In [0]:
model_full_name = dbutils.widgets.get("model_full_name")
project_path = dbutils.widgets.get("project_path")
catalog = model_full_name.split('.')[0]
schema = model_full_name.split('.')[1]
model_name = model_full_name.split('.')[-1]
feature_table_name = f"{catalog}.{schema}.churn_user_features"
target_col = "churn"
pos_label = 1
metric_name = "f1"

#xp_name = f"{model_name}_Exp_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S')}"
xp_name = f"{model_name}_Exp"
xp_path = f"{project_path}/Experiments"
experiment_full_path = f"{xp_path}/{xp_name}"
run_name=xp_name

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"""USE `{catalog}`.`{schema}`""")    

DataFrame[]

In [0]:
try:
  from databricks.sdk import WorkspaceClient
  w = WorkspaceClient()
  r = w.workspace.mkdirs(path=xp_path)
except Exception as e:
  print(f"ERROR: No puedo crear folder para los experimentos: {xp_path} - Por favor cree la carpeta manualmente. Mensaje: {e}")
  raise e

## 2. Conjuntos de datos y Caracteristicas

In [0]:
df = spark.table(feature_table_name).sample(fraction=0.5, seed=42)
display(df.limit(5))

age_group,canal,churn,country,gender,user_id,ingest_datetime_silver,platform,event_count,session_count,order_count,total_amount,total_item,last_transaction,days_since_creation,days_since_last_activity,days_last_event
4,PHONE,1,SPAIN,0,f617a497-9026-445d-b077-f06f2e3aa82c,2026-04-10T21:26:14.553Z,ios,4,3,4,223,9,2023-06-05T06:23:07.000Z,1185,1039,1039
1,MOBILE,0,FR,1,ee8a18a8-eebf-40cd-a46d-5a2474089c64,2026-04-10T21:26:14.553Z,android,6,5,6,247,10,2023-06-09T06:33:02.000Z,1140,1036,1039
5,PHONE,1,USA,1,5f52e274-e0cd-4665-9c15-05590385692c,2026-04-10T21:26:14.553Z,other,1,0,1,96,3,2023-06-08T02:08:17.000Z,1377,1043,1041
7,MOBILE,1,USA,1,35bd5a18-66a5-482c-b618-e353128e67fe,2026-04-10T21:26:14.553Z,other,4,3,4,191,7,2023-06-09T18:28:14.000Z,1559,1039,1036
5,WEBAPP,0,SPAIN,1,509b3da6-c25a-4b9a-b375-5ade442b95c2,2026-04-10T21:26:14.553Z,ios,2,1,2,103,5,2023-06-08T21:50:54.000Z,1489,1041,1038


In [0]:
supported_cols = ["gender", "last_transaction", "canal", "event_count", "days_since_creation", "country", "session_count",
    "order_count", "days_last_event", "days_since_last_activity",
    "total_amount", "total_item", "age_group", "platform"
]
df = df.select(*supported_cols, target_col)

train_df,  test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Training: {train_df.count()}, Testing: {test_df.count()}")

Training: 26607, Testing: 6737


## 3. Entrenamiento
Iniciar AutoML.

In [0]:
from databricks import automl

summary = automl.classify(train_df, target_col=target_col, pos_label=pos_label, experiment_name=xp_name, 
                          data_dir=experiment_full_path, timeout_minutes=30, max_trials=10)
print(f"Best trial notebook: {summary.best_trial.notebook_path}")

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-6974455923561611>, line 1
----> 1 from databricks import automl
      3 summary = automl.classify(train_df, target_col=target_col, pos_label=pos_label, experiment_name=xp_name, 
      4                           data_dir=experiment_full_path, timeout_minutes=30, max_trials=10)
      5 print(f"Best trial notebook: {summary.best_trial.notebook_path}")

ImportError: cannot import name 'automl' from 'databricks' (/databricks/python_shell/lib/databricks/__init__.py)

In [0]:
help(sumary)